# Deep Weakly-Supervised Anomaly Detection — PReNet Replication

Replication of **PReNet** (Pang et al., KDD '23) on the Annthyroid dataset, including environment porting from TensorFlow 1.14/Python 3.6 to TF 2.x/Python 3.12, a pairwise-supervision ablation, and a training-loss visualization.

**Author:** Tejvansh Singh Randhawa

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
%cd /content/drive/MyDrive/Term_Project/Code
!git clone https://github.com/mala-lab/PReNet.git


In [ ]:
%cd /content/drive/MyDrive/Term_Project/Code/PReNet
!ls


## 2. Compatibility Patches

The official codebase targets TensorFlow 1.14 / Python 3.6. Running it on TF 2.19 / Python 3.12 required patching three breaking API changes.

### 2.1 Model checkpoint filename (`.h5` → `.weights.h5`)

In [ ]:
with open('prenet.py', 'r') as f:
    content = f.read()

content = content.replace(
    'str(network_depth) +"d.h5"',
    'str(network_depth) +"d.weights.h5"'
)

with open('prenet.py', 'w') as f:
    f.write(content)

print("Fix 1 done: checkpoint filename.")


### 2.2 `model.fit_generator()` → `model.fit()` with `tf.data.Dataset`

`fit_generator` was removed in TF 2.x. The custom pair-sampling generator is wrapped in a `tf.data.Dataset` instead.

In [ ]:
with open('prenet.py', 'r') as f:
    content = f.read()

old_fit = (
    "                history = model.fit_generator(pair_generator(x_train, outlier_indices, inlier_indices, Y, batch_size, nb_batch, rng),\n"
    "                                    steps_per_epoch = nb_batch,\n"
    "                                    epochs = epochs,\n"
    "                                    callbacks=[checkpointer])            \n"
)

new_fit = (
    "                def make_dataset2():\n"
    "                    def gen():\n"
    "                        rng_inner = np.random.RandomState(42)\n"
    "                        for _ in range(nb_batch):\n"
    "                            s1, s2, lbls = pair_batch_generation(x_train, outlier_indices, inlier_indices, Y, batch_size, rng_inner)\n"
    "                            yield (s1.astype(np.float32), s2.astype(np.float32)), lbls.astype(np.float32)\n"
    "                    dim = x_train.shape[1]\n"
    "                    return tf.data.Dataset.from_generator(\n"
    "                        gen,\n"
    "                        output_signature=(\n"
    "                            (tf.TensorSpec(shape=(batch_size, dim), dtype=tf.float32),\n"
    "                             tf.TensorSpec(shape=(batch_size, dim), dtype=tf.float32)),\n"
    "                            tf.TensorSpec(shape=(batch_size,), dtype=tf.float32)\n"
    "                        )\n"
    "                    )\n"
    "                history = model.fit(make_dataset2(),\n"
    "                                    epochs = epochs,\n"
    "                                    callbacks=[checkpointer])\n"
)

content = content.replace(old_fit, new_fit)

with open('prenet.py', 'w') as f:
    f.write(content)

print("Fix 2 done: fit_generator -> tf.data.Dataset.")


### 2.3 `keras.backend` loss function → `tf.reduce_mean`

In [ ]:
with open('prenet.py', 'r') as f:
    content = f.read()

# Fix keras backend imports
content = content.replace(
    'from keras import backend as K',
    'import tensorflow as tf\nfrom keras import backend as K'
)

# Fix regression_loss function
content = content.replace(
    'def regression_loss(y_true, y_pred):\n    return K.mean(K.abs(y_pred - y_true), axis=-1)',
    'def regression_loss(y_true, y_pred):\n    return tf.reduce_mean(tf.abs(y_pred - y_true), axis=-1)'
)

with open('prenet.py', 'w') as f:
    f.write(content)

print("Fix 3 done: regression_loss now uses tf.reduce_mean.")


## 3. Main Experiment

PReNet on Annthyroid (7,200 instances, 21 features, 7.4% anomaly rate), K=30 known outliers, 5 seeds.

In [ ]:
!python prenet.py \\
  --data_set annthyroid_21feat_normalised \\
  --data_format 0 \\
  --network_depth 2 \\
  --known_outliers 30 \\
  --cont_rate 0.02 \\
  --epochs 50 \\
  --nb_batch 20 \\
  --batch_size 512 \\
  --ensemble_size 1 \\
  --runs 5 \\
  --h_lambda 0.01 \\
  --ordinal_labels 0,4,8 \\
  --input_path data/ \\
  --output /content/drive/MyDrive/Term_Project/results/prenet_annthyroid.csv


In [ ]:
import os
os.makedirs('/content/drive/MyDrive/Term_Project/results', exist_ok=True)

!cat /content/drive/MyDrive/Term_Project/results/prenet_annthyroid.csv


### Paper vs. Replication comparison

In [ ]:
import pandas as pd
import os

os.makedirs('/content/drive/MyDrive/Term_Project/results', exist_ok=True)

comparison = pd.DataFrame({
    'Method': ['Paper (PReNet, K=60)', 'Our Replication (K=30)', 'Difference'],
    'AUC-ROC': [0.781, 0.8012, round(0.8012 - 0.781, 4)],
    'AUC-PR': [0.298, 0.2918, round(0.2918 - 0.298, 4)],
    'Runs': [10, 5, '-'],
    'Known Outliers (K)': [60, 30, '-'],
    'Notes': [
        'Original paper Table 2 results',
        'Our replication mean over 5 seeds',
        'Positive = our result is higher'
    ]
})

comparison.to_csv('/content/drive/MyDrive/Term_Project/results/comparison_paper_vs_ours.csv', index=False)
print("Saved successfully!")
print(comparison.to_string(index=False))


## 4. Ablation Study

Same encoder architecture, but trained with plain binary cross-entropy instead of pairwise relation supervision — isolates the contribution of the pairwise formulation.

In [ ]:
%%writefile /content/drive/MyDrive/Term_Project/Code/ablation.py

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from keras.models import Model
from keras.layers import Input, Dense
from keras.optimizers import RMSprop
from keras import regularizers
import tensorflow as tf

# Load data
data = pd.read_csv('/content/drive/MyDrive/Term_Project/Code/PReNet/data/annthyroid_21feat_normalised.csv')
print("Data shape:", data.shape)

X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

def build_model(input_dim):
    x_input = Input(shape=(input_dim,))
    hidden = Dense(20, activation='relu',
                   kernel_regularizer=regularizers.l2(0.01))(x_input)
    output = Dense(1, activation='sigmoid')(hidden)
    model = Model(x_input, output)
    model.compile(loss='binary_crossentropy',
                  optimizer=RMSprop(clipnorm=1.))
    return model

seeds = [0, 1, 2, 3, 4]
known_outliers = 30
aucs = []
aps = []

for seed in seeds:
    np.random.seed(seed)
    tf.random.set_seed(seed)

    x_train, x_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y)

    outlier_indices = np.where(y_train == 1)[0]
    n_outliers = len(outlier_indices)

    # Keep only K known outliers
    rng = np.random.RandomState(seed)
    if n_outliers > known_outliers:
        remove_idx = rng.choice(outlier_indices,
                                n_outliers - known_outliers,
                                replace=False)
        x_train = np.delete(x_train, remove_idx, axis=0)
        y_train = np.delete(y_train, remove_idx, axis=0)

    print(f"Seed {seed} | Train size: {x_train.shape[0]}, Outliers: {int(y_train.sum())}")

    model = build_model(x_train.shape[1])
    model.fit(x_train, y_train,
              epochs=50,
              batch_size=512,
              verbose=0)

    scores = model.predict(x_test).flatten()
    auc = roc_auc_score(y_test, scores)
    ap = average_precision_score(y_test, scores)
    aucs.append(auc)
    aps.append(ap)
    print(f"Seed {seed} | AUC-ROC: {auc:.4f}, AUC-PR: {ap:.4f}")

print(f"\nAblation Mean AUC-ROC: {np.mean(aucs):.4f} +/- {np.std(aucs):.4f}")
print(f"Ablation Mean AUC-PR:  {np.mean(aps):.4f} +/- {np.std(aps):.4f}")

# Save results
ablation = pd.DataFrame({'seed': seeds, 'auc_roc': aucs, 'auc_pr': aps})
ablation.to_csv('/content/drive/MyDrive/Term_Project/results/ablation_results.csv', index=False)
print("\nAblation results saved.")


In [ ]:
%cd /content/drive/MyDrive/Term_Project/Code
!python ablation.py


## 5. Training Loss Curve

Reimplements the PReNet pair-sampling and training loop (seed 0) to log per-epoch loss for visualization.

In [ ]:
%%writefile /content/drive/MyDrive/Term_Project/Code/training_curves.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from keras.models import Model
from keras.layers import Input, Dense, concatenate
from keras.optimizers import RMSprop
from keras import regularizers
import tensorflow as tf

# Load data
data = pd.read_csv('/content/drive/MyDrive/Term_Project/Code/PReNet/data/annthyroid_21feat_normalised.csv')
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

def regression_loss(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_pred - y_true), axis=-1)

def build_prenet(input_dim):
    x_input = Input(shape=(input_dim,))
    intermediate = Dense(20, activation='relu',
                         kernel_regularizer=regularizers.l2(0.01))(x_input)
    base_network = Model(x_input, intermediate)
    input_a = Input(shape=(input_dim,))
    input_b = Input(shape=(input_dim,))
    processed_a = base_network(input_a)
    processed_b = base_network(input_b)
    merged = concatenate([processed_a, processed_b])
    score = Dense(1, activation='linear')(merged)
    model = Model([input_a, input_b], score)
    model.compile(loss=regression_loss, optimizer=RMSprop(clipnorm=1.))
    return model

def make_pairs(x_train, outlier_indices, inlier_indices, batch_size, nb_batch, seed):
    uu, au, aa = 0, 4, 8
    rng = np.random.RandomState(seed)
    all_s1, all_s2, all_labels = [], [], []
    for _ in range(nb_batch):
        dim = x_train.shape[1]
        pairs1 = np.empty((batch_size, dim))
        pairs2 = np.empty((batch_size, dim))
        labels = []
        n_in = len(inlier_indices)
        n_out = len(outlier_indices)
        block = batch_size // 4
        sid = rng.choice(n_in, block*4, replace=False)
        pairs1[0:block*2] = x_train[inlier_indices[sid[0:block*2]]]
        pairs2[0:block*2] = x_train[inlier_indices[sid[block*2:block*4]]]
        labels += block*2*[uu]
        sid = rng.choice(n_in, block, replace=False)
        pairs1[block*2:block*3] = x_train[inlier_indices[sid]]
        sid = rng.choice(n_out, block)
        pairs2[block*2:block*3] = x_train[outlier_indices[sid]]
        labels += block*[au]
        for i in range(block*3, batch_size):
            sid = rng.choice(n_out, 2, replace=False)
            pairs1[i] = x_train[outlier_indices[sid[0]]]
            pairs2[i] = x_train[outlier_indices[sid[1]]]
            labels += [aa]
        all_s1.append(pairs1)
        all_s2.append(pairs2)
        all_labels.append(np.array(labels, dtype=float))
    return (np.concatenate(all_s1).astype(np.float32),
            np.concatenate(all_s2).astype(np.float32),
            np.concatenate(all_labels).astype(np.float32))

# Use seed 0 for the plot
np.random.seed(0)
tf.random.set_seed(0)
known_outliers = 30
epochs = 50
nb_batch = 20
batch_size = 512

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

outlier_indices = np.where(y_train == 1)[0]
rng = np.random.RandomState(0)
if len(outlier_indices) > known_outliers:
    remove_idx = rng.choice(outlier_indices,
                            len(outlier_indices) - known_outliers,
                            replace=False)
    x_train = np.delete(x_train, remove_idx, axis=0)
    y_train = np.delete(y_train, remove_idx, axis=0)

outlier_indices = np.where(y_train == 1)[0]
inlier_indices  = np.where(y_train == 0)[0]

model = build_prenet(x_train.shape[1])

loss_history = []
print("Training for plotting...")
for epoch in range(epochs):
    s1, s2, labels = make_pairs(x_train, outlier_indices,
                                inlier_indices, batch_size, nb_batch, epoch)
    hist = model.fit([s1, s2], labels,
                     batch_size=batch_size,
                     epochs=1, verbose=0)
    loss_history.append(hist.history['loss'][0])
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}/50 | Loss: {loss_history[-1]:.4f}")

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, epochs+1), loss_history, color='#2E4057', linewidth=2)
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Training Loss (MAE)', fontsize=12)
ax.set_title('PReNet Training Loss on Annthyroid (Seed 0)', fontsize=13)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/Term_Project/results/training_curve.png', dpi=150)
plt.show()
print("Plot saved.")


In [ ]:
%cd /content/drive/MyDrive/Term_Project/Code
!python training_curves.py


In [ ]:
from IPython.display import Image
Image('/content/drive/MyDrive/Term_Project/results/training_curve.png')
